# Instal Dependensi


In [1]:
!pip install --quiet pandas numpy scikit-learn matplotlib tqdm


# Konfigurasi & load dataset

In [2]:
INPUT_CSV = "data_bandara_singapura.csv"
OUTPUT_DIR = "outputs_sentiment"
SAMPLE_FOR_TRAIN = None


import os, re, json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load
assert os.path.exists(INPUT_CSV), f"Tidak menemukan file: {INPUT_CSV}"
df = pd.read_csv(INPUT_CSV)

# Deteksi kolom teks
text_col = None
for c in df.columns:
    if c.lower() in ["text","comment","comments","content","body"]:
        text_col = c
        break
assert text_col is not None, f"Tidak menemukan kolom komentar. Kolom tersedia: {list(df.columns)}"
print("Kolom komentar terdeteksi:", text_col, "| Jumlah baris:", len(df))


Kolom komentar terdeteksi: text | Jumlah baris: 10000


# Preprocessing teks

In [8]:
def clean_text(s: str) -> str:
    if not isinstance(s,str): return ""
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)       # URL
    s = re.sub(r"[@#]\w+", " ", s)  # hapus atau ubah komentarnya, misal:  # hapus mention dan hashtag
    s = re.sub(r"&\w+;", " ", s)                  # html entities
    s = re.sub(r"[^a-z0-9'\-\.\,\!\?\s]", " ", s) # keep basic chars/punct
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_clean"] = df[text_col].astype(str).apply(clean_text)
df = df[df["text_clean"].str.len() > 0].copy()
print("Setelah cleaning:", len(df), "baris")
df[["text_clean"]].head(3)


Setelah cleaning: 9993 baris


,text_clean
0,4th super excited fantastic
1,"nice video, also i am excited for samchui airl..."
2,omg!! i am from and live in hong kong!


# Ekstraksi fitur & 3 skema pelatihan

In [4]:
# ===================== IMPROVED AUTO-LABEL + TRAINING (3 SCHEMES) =====================
import os, re, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -------- 1) AUTO-LABEL: domain lexicon + phrase rules + negation + intensifier --------

NEG_PHRASES = [
    r"lost (my|the|our)\s+luggage",
    r"(lost|missing|mishandled)\s+(bag|baggage|luggage)",
    r"worst (flight|experience)",
    r"delayed\s+(for\s+)?(hours?|so long|too long)",
    r"(flight|flights?)\s+cancell?ed",
    r"overbooked",
    r"no\s+compensation",
    r"poor\s+communication",
    r"rude\s+(staff|crew)",
    r"missed\s+connection",
]
POS_PHRASES = [
    r"best (flight|experience)",
    r"highly\s+recommended",
    r"cabin\s+crew\s+(were\s+)?(very\s+)?(friendly|amazing|excellent)",
    r"(very\s+)?comfortable\s+seats?",
    r"on(\-|\s*)time",
    r"great\s+service",
]


POS = set("""
amazing awesome great excellent fantastic superb outstanding wonderful perfect
pleasant smooth comfortable friendly polite helpful attentive professional clean
spacious delicious enjoyable recommend recommended satisfied satisfying premium
on-time ontime punctual quick fast efficient upgrade upgraded comfy love loved
nice lovely beautiful impressive best brilliant kudos wow thank you thanks
""".split())

NEG = set("""
bad worse worst terrible horrible awful dirty cramped uncomfortable rude impolite
unhelpful unprofessional delay delayed delays late cancelled canceled cancel
lost missing broken damaged disgusting smelly noisy messy slow inefficient
poor nightmare hate hated angry upset disappointed disappointing complaint
queue queues long wait waiting expensive overpriced scam
""".split())

NEGATORS = set("not never no hardly barely scarcely seldom rarely without".split())
INTENSIFIERS = set("very extremely really so too quite highly super".split())

def clean_text_basic(s: str) -> str:
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[@#]\w+", " ", s)
    s = re.sub(r"&\w+;", " ", s)
    s = re.sub(r"[^a-z0-9'\-\.\,\!\?\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def toks(s):
    return [w for w in re.findall(r"[a-z0-9']+", s) if w]

def phrase_boost(text: str) -> int:
    """Deteksi frasa kuat; berikan skor besar agar tidak kalah oleh noise."""
    sc = 0
    for pat in NEG_PHRASES:
        if re.search(pat, text):
            sc -= 3
    for pat in POS_PHRASES:
        if re.search(pat, text):
            sc += 3
    return sc

def token_score(tokens):
    """Lexicon scoring dengan negation + intensifier (memperkuat ±2 jika ada intensifier dekat)."""
    score, window_neg, recent_neg = 0, 3, 0
    for i, t in enumerate(tokens):
        if t in NEGATORS:
            recent_neg = window_neg
            continue

        val = 1 if t in POS else (-1 if t in NEG else 0)


        intens = 0
        if i > 0 and tokens[i-1] in INTENSIFIERS:
            intens += 1
        if i+1 < len(tokens) and tokens[i+1] in INTENSIFIERS:
            intens += 1
        if intens > 0 and val != 0:
            val = val * (1 + intens)


        if recent_neg > 0:
            val = -val
            recent_neg -= 1

        score += val
        if recent_neg > 0 and t not in POS and t not in NEG:
            recent_neg -= 1
    return score

def label_rule(text: str, pos_thr=2, neg_thr=-2):
    text_c = clean_text_basic(text)
    sc = phrase_boost(text_c)
    sc += token_score(toks(text_c))
    if sc >= pos_thr:
        return "positive", sc
    if sc <= neg_thr:
        return "negative", sc
    return "neutral", sc


if "text_clean" not in df.columns:

    text_col = None
    for c in df.columns:
        if c.lower() in ["text","comment","comments","content","body"]:
            text_col = c
            break
    assert text_col is not None, f"Tidak menemukan kolom komentar. Kolom tersedia: {list(df.columns)}"
    df["text_clean"] = df[text_col].astype(str).apply(clean_text_basic)


labels_scores = df["text_clean"].apply(lambda s: label_rule(s))
df["label"] = labels_scores.apply(lambda x: x[0])
df["_rb_score"] = labels_scores.apply(lambda x: x[1])

print("Distribusi label (rule-based):")
print(df["label"].value_counts())

# -------- 2) DATA TRAIN/TEST PREP --------
if 'SAMPLE_FOR_TRAIN' not in globals():
    SAMPLE_FOR_TRAIN = None

if SAMPLE_FOR_TRAIN is not None and SAMPLE_FOR_TRAIN < len(df):
    frac = SAMPLE_FOR_TRAIN / len(df)
    df_train = df.groupby("label", group_keys=False).apply(lambda x: x.sample(frac=min(1.0, frac), random_state=42))
    df_train = df_train.sample(min(SAMPLE_FOR_TRAIN, len(df_train)), random_state=42)
else:
    df_train = df

X_all = df_train["text_clean"].values
y_all = df_train["label"].values
print("Dipakai untuk training:", len(df_train))

# -------- 3) VECTORIZER KUAT  --------
word_vec = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1,3),
    min_df=1,
    max_df=0.99,
    sublinear_tf=True
)
char_vec = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=2,
    max_df=0.99,
    sublinear_tf=True
)


word_vec.fit(X_all)
char_vec.fit(X_all)

def transform_both(texts):
    return hstack([word_vec.transform(texts), char_vec.transform(texts)])

# -------- 4) EVALUATOR + CONFUSION MATRIX PLOTTER --------
def eval_and_plot(name, clf, X_raw, y, test_size, random_state, cm_path_png):
    Xtr_raw, Xte_raw, ytr, yte = train_test_split(
        X_raw, y, test_size=test_size, random_state=random_state, stratify=y
    )
    Xtr = transform_both(Xtr_raw)
    Xte = transform_both(Xte_raw)
    clf.fit(Xtr, ytr)
    yhat = clf.predict(Xte)
    acc = accuracy_score(yte, yhat)
    rep = classification_report(yte, yhat, digits=4)
    cm = confusion_matrix(yte, yhat, labels=["negative","neutral","positive"])
    # plot CM
    fig = plt.figure(figsize=(5,4))
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"Confusion Matrix - {name}")
    plt.colorbar()
    ticks = np.arange(3)
    plt.xticks(ticks, ["negative","neutral","positive"], rotation=45)
    plt.yticks(ticks, ["negative","neutral","positive"])
    thr = cm.max()/2.
    for i,j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", color="white" if cm[i, j] > thr else "black")
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(cm_path_png, dpi=160, bbox_inches='tight')
    plt.close(fig)
    return acc, rep, (Xtr, ytr), (Xte, yte), (Xtr_raw, Xte_raw)

metrics_text = []

# -------- 5) SCHEME 1: LinearSVC (balanced), 80/20 --------
clf1 = LinearSVC(class_weight="balanced", random_state=42)
acc1, rep1, tr1, te1, raw1 = eval_and_plot(
    "LinearSVC (80/20)", clf1, X_all, y_all, 0.2, 42, os.path.join(OUTPUT_DIR, "cm_scheme1.png")
)
metrics_text.append(f"=== Scheme 1: LinearSVC (80/20) ===\nAccuracy: {acc1:.4f}\n{rep1}\nSaved CM: {os.path.join(OUTPUT_DIR, 'cm_scheme1.png')}")

# -------- 6) SCHEME 2: Logistic Regression (balanced), 80/20 --------
clf2 = LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=None)
acc2, rep2, tr2, te2, raw2 = eval_and_plot(
    "LogReg (80/20)", clf2, X_all, y_all, 0.2, 123, os.path.join(OUTPUT_DIR, "cm_scheme2.png")
)
metrics_text.append(f"=== Scheme 2: LogisticRegression (80/20) ===\nAccuracy: {acc2:.4f}\n{rep2}\nSaved CM: {os.path.join(OUTPUT_DIR, 'cm_scheme2.png')}")

# -------- 7) SCHEME 3: RandomForest, 70/30 --------
clf3 = RandomForestClassifier(n_estimators=350, n_jobs=-1, random_state=2025)
acc3, rep3, tr3, te3, raw3 = eval_and_plot(
    "RandomForest (70/30)", clf3, X_all, y_all, 0.3, 2025, os.path.join(OUTPUT_DIR, "cm_scheme3.png")
)
metrics_text.append(f"=== Scheme 3: RandomForest (70/30) ===\nAccuracy: {acc3:.4f}\n{rep3}\nSaved CM: {os.path.join(OUTPUT_DIR, 'cm_scheme3.png')}")

print("Akurasi:", {"LinearSVC": round(acc1,4), "LogReg": round(acc2,4), "RandomForest": round(acc3,4)})

# -------- 8) BEST PIPE + HYBRID FALLBACK --------

accs = [("svm", acc1, clf1), ("logreg", acc2, clf2), ("rf", acc3, clf3)]
accs.sort(key=lambda x: x[1], reverse=True)
best_name, best_acc, best_clf = accs[0]
print("Best model:", best_name, "| acc:", round(best_acc,4))

class BestPipe:
    def __init__(self, wv, cv, model):
        self.wv = wv
        self.cv = cv
        self.model = model
    def predict(self, texts):
        Xt = hstack([self.wv.transform(texts), self.cv.transform(texts)])
        return self.model.predict(Xt)

best_pipe = BestPipe(word_vec, char_vec, best_clf)

def hybrid_predict(texts, model, pos_thr=2, neg_thr=-2):
    preds = model.predict(texts)
    final = []
    for t, p in zip(texts, preds):
        lbl, sc = label_rule(t, pos_thr=pos_thr, neg_thr=neg_thr)  # skor rule-based
        if p == "neutral" and sc <= -2:
            final.append("negative")
        elif p == "neutral" and sc >= 2:
            final.append("positive")
        else:
            final.append(p)
    return np.array(final)



Distribusi label (rule-based):
label
neutral     8357
positive    1488
negative     148
Name: count, dtype: int64
Dipakai untuk training: 9993


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Akurasi: {'LinearSVC': 0.916, 'LogReg': 0.9175, 'RandomForest': 0.8753}
Best model: logreg | acc: 0.9175


In [5]:
from sklearn.metrics import accuracy_score
import numpy as np

def percent_distribution(y_pred):
    labels = ["negative", "neutral", "positive"]
    n = len(y_pred)
    out = {}
    for l in labels:
        out[l] = round(100.0 * np.sum(np.array(y_pred) == l) / max(1, n), 2)
    return out

# --- Evaluasi untuk masing-masing skema ---
Xtr1, Xte1, ytr1, yte1 = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# Skema 1
y_pred1 = clf1.fit(transform_both(Xtr1), ytr1).predict(transform_both(Xte1))
print("=== Scheme 1: LinearSVC (80/20) ===")
print("Akurasi:", round(accuracy_score(yte1, y_pred1)*100, 2), "%")
print("Distribusi Prediksi:", percent_distribution(y_pred1), "\n")

# Skema 2
y_pred2 = clf2.fit(transform_both(Xtr1), ytr1).predict(transform_both(Xte1))
print("=== Scheme 2: Logistic Regression (80/20) ===")
print("Akurasi:", round(accuracy_score(yte1, y_pred2)*100, 2), "%")
print("Distribusi Prediksi:", percent_distribution(y_pred2), "\n")

# Skema 3
y_pred3 = clf3.fit(transform_both(Xtr1), ytr1).predict(transform_both(Xte1))
print("=== Scheme 3: RandomForest (80/30) ===")
print("Akurasi:", round(accuracy_score(yte1, y_pred3)*100, 2), "%")
print("Distribusi Prediksi:", percent_distribution(y_pred3))

=== Scheme 1: LinearSVC (80/20) ===
Akurasi: 91.6 %
Distribusi Prediksi: {'negative': np.float64(0.05), 'neutral': np.float64(87.24), 'positive': np.float64(12.71)} 

=== Scheme 2: Logistic Regression (80/20) ===
Akurasi: 91.35 %
Distribusi Prediksi: {'negative': np.float64(0.8), 'neutral': np.float64(81.89), 'positive': np.float64(17.31)} 

=== Scheme 3: RandomForest (80/30) ===
Akurasi: 86.84 %
Distribusi Prediksi: {'negative': np.float64(0.0), 'neutral': np.float64(95.9), 'positive': np.float64(4.1)}


In [6]:
FULL_LABELED_CSV = os.path.join(OUTPUT_DIR, "data_bandara_singapura_labeled_full.csv")
df.to_csv(FULL_LABELED_CSV, index=False, encoding="utf-8")
print("Full labeled CSV saved to:", FULL_LABELED_CSV)


Full labeled CSV saved to: outputs_sentiment/data_bandara_singapura_labeled_full.csv


# Simpan laporan metrik

In [7]:
# -------- 9) Simpan metrics --------
METRICS_TXT = os.path.join(OUTPUT_DIR, "metrics_report.txt")
with open(METRICS_TXT, "w", encoding="utf-8") as f:
    f.write("\n\n".join(metrics_text))
print("Saved:", METRICS_TXT)



Saved: outputs_sentiment/metrics_report.txt


In [9]:
 pip freeze requirements.txt

absl-py==1.4.0
absolufy-imports==0.3.1
accelerate==1.10.1
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.0
aiosignal==1.4.0
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.17.0
altair==5.5.0
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.11.0
anywidget==0.9.18
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.1
arrow==1.3.0
arviz==0.22.0
astropy==7.1.1
astropy-iers-data==0.2025.10.13.0.37.17
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.0.1
Authlib==1.6.5
autograd==1.8.0
babel==2.17.0
backcall==0.2.0
beartype==0.22.2
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.24.0
bigquery-magics==0.10.3
bleach==6.2.0
blinker==1.9.0
blis==1.3.0
blobfile==3.1.0
blosc2==3.10.1
bokeh==3.7.3
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
Brotli==1.1.0
build==1.3.0
CacheControl==0.14.3
cachetools==5.5.2
catalogue==2.0.10
certifi==2025.10.5
cffi==2.0.0
chardet==5.2.0
charset-normalizer==3.4.4
chex==0.1.90
cl